# IMDb Hybrid Recommender Training (Colab)
Trains multiple configurations, picks the best model, and exports artifacts + metrics.

In [ ]:
!pip -q install kagglehub

In [ ]:
import json
import os
import pickle
import re

import kagglehub
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

In [ ]:
def safe_filename(title, year):
    cleaned = re.sub(r"[^A-Za-z0-9]+", "_", str(title).strip()).strip("_")
    return f"{cleaned}_{int(year)}.jpg"

def genre_set(value):
    return {g.strip().lower() for g in str(value).split(",") if g.strip()}

def build_features(df, ngram_range=(1,2), min_df=1, max_features=None):
    df = df.copy()
    df["poster"] = df.apply(lambda r: safe_filename(r["primaryTitle"], r["startYear"]), axis=1)
    df["text_features"] = (
        df["genres"].str.lower().str.replace(",", " ", regex=False) + " " +
        df["directors"].str.lower().str.strip() + " " +
        df["writers"].str.lower().str.strip()
    )
    numeric = df[["runtimeMinutes", "numVotes", "averageRating", "startYear"]].copy()
    scaler = MinMaxScaler()
    numeric_scaled = scaler.fit_transform(numeric)
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=max_features,
    )
    text_vectors = vectorizer.fit_transform(df["text_features"])
    X = hstack([text_vectors, csr_matrix(numeric_scaled)])
    return X, vectorizer, scaler

def train_model(X, n_neighbors=20):
    model = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
    model.fit(X)
    return model

def evaluate_model(df_eval, X_eval, model_eval, k=5):
    n = X_eval.shape[0]
    if n == 0:
        return {
            "movies_used": 0,
            "k": k,
            "avg_neighbor_similarity_at_k": 0.0,
            "avg_genre_jaccard_at_k": 0.0,
            "catalog_coverage_at_k": 0.0,
            "precision_at_k": 0.0,
            "recall_at_k": 0.0,
            "f1_score_at_k": 0.0,
        }

    sims, jaccards = [], []
    recommended_indices = set()
    precision_vals, recall_vals = [], []
    all_genres = [genre_set(g) for g in df_eval["genres"]]

    for i in range(n):
        distances, indices = model_eval.kneighbors(X_eval[i], n_neighbors=min(k + 1, n))
        rec_idxs = [idx for idx in indices[0] if idx != i][:k]
        rec_dists = [dist for idx, dist in zip(indices[0], distances[0]) if idx != i][:k]
        sims.extend([1.0 - float(d) for d in rec_dists])
        recommended_indices.update(rec_idxs)
        src = all_genres[i]
        relevant_pool = {j for j in range(n) if j != i and len(src & all_genres[j]) > 0}
        hits = sum(1 for ridx in rec_idxs if len(src & all_genres[ridx]) > 0)
        precision_vals.append(hits / max(1, len(rec_idxs)))
        recall_vals.append(hits / max(1, len(relevant_pool)))
        for ridx in rec_idxs:
            tgt = all_genres[ridx]
            union = len(src | tgt)
            inter = len(src & tgt)
            jaccards.append((inter / union) if union else 0.0)

    precision_at_k = float(np.mean(precision_vals)) if precision_vals else 0.0
    recall_at_k = float(np.mean(recall_vals)) if recall_vals else 0.0
    f1_score_at_k = (2 * precision_at_k * recall_at_k / (precision_at_k + recall_at_k)) if (precision_at_k + recall_at_k) > 0 else 0.0

    return {
        "movies_used": int(n),
        "k": int(k),
        "avg_neighbor_similarity_at_k": float(np.mean(sims)) if sims else 0.0,
        "avg_genre_jaccard_at_k": float(np.mean(jaccards)) if jaccards else 0.0,
        "catalog_coverage_at_k": float(len(recommended_indices) / n),
        "precision_at_k": float(precision_at_k),
        "recall_at_k": float(recall_at_k),
        "f1_score_at_k": float(f1_score_at_k),
    }

def score_metrics(m):
    return 0.40 * m["f1_score_at_k"] + 0.30 * m["avg_neighbor_similarity_at_k"] + 0.20 * m["avg_genre_jaccard_at_k"] + 0.10 * m["catalog_coverage_at_k"]


In [ ]:
# Download + clean data
path = kagglehub.dataset_download("tiagoadrianunes/imdb-top-5000-movies")
files = os.listdir(path)
df = pd.read_csv(os.path.join(path, files[0]))

df = df[[
    "primaryTitle", "genres", "directors", "writers",
    "averageRating", "startYear", "runtimeMinutes", "numVotes"
]]
df = df[(df["startYear"] >= 2023) & (df["startYear"] <= 2026)]
df = df.dropna().drop_duplicates()
df = df[df["numVotes"] > 10000].reset_index(drop=True)

print("Dataset path:", path)
print("Movies after cleaning:", len(df))

In [ ]:
# Hyperparameter search and best-model selection
search_space = [
    {"ngram_range": (1, 1), "min_df": 1, "max_features": None, "n_neighbors": 20},
    {"ngram_range": (1, 2), "min_df": 1, "max_features": None, "n_neighbors": 20},
    {"ngram_range": (1, 2), "min_df": 2, "max_features": None, "n_neighbors": 20},
    {"ngram_range": (1, 2), "min_df": 1, "max_features": 15000, "n_neighbors": 20},
    {"ngram_range": (1, 2), "min_df": 2, "max_features": 20000, "n_neighbors": 25},
    {"ngram_range": (1, 3), "min_df": 2, "max_features": 25000, "n_neighbors": 25},
]

best = None
for i, cfg in enumerate(search_space, start=1):
    X_try, vectorizer_try, scaler_try = build_features(
        df,
        ngram_range=cfg["ngram_range"],
        min_df=cfg["min_df"],
        max_features=cfg["max_features"],
    )
    model_try = train_model(X_try, n_neighbors=cfg["n_neighbors"])
    metrics_try = evaluate_model(df, X_try, model_try, k=5)
    score_try = score_metrics(metrics_try)

    print(f"[{i}/{len(search_space)}] {cfg}")
    print(f"  metrics: {metrics_try}")
    print(f"  score:   {score_try:.6f}")

    if best is None or score_try > best["score"]:
        best = {
            "cfg": cfg,
            "X": X_try,
            "vectorizer": vectorizer_try,
            "scaler": scaler_try,
            "model": model_try,
            "metrics": metrics_try,
            "score": score_try,
        }

print("\nBest config:", best["cfg"])

In [ ]:
# Build recommendations and export files
df_out = df.copy()
df_out = df_out.rename(columns={
    "primaryTitle": "title",
    "averageRating": "rating",
    "startYear": "year",
})
df_out["poster"] = df_out.apply(lambda r: safe_filename(r["title"], r["year"]), axis=1)

recommendations = []
for i in range(best["X"].shape[0]):
    distances, indices = best["model"].kneighbors(best["X"][i], n_neighbors=6)
    rec_titles = [df_out.iloc[idx]["title"] for idx in indices[0] if idx != i][:5]
    recommendations.append(rec_titles)
df_out["recommendations"] = recommendations

pickle.dump(best["model"], open("movie_model_hybrid.pkl", "wb"))
pickle.dump(best["vectorizer"], open("vectorizer_hybrid.pkl", "wb"))
pickle.dump(best["scaler"], open("scaler_hybrid.pkl", "wb"))

with open("movies.json", "w", encoding="utf-8") as f:
    json.dump({"movies": df_out.to_dict(orient="records")}, f, indent=2)

metrics_output = dict(best["metrics"])
metrics_output["search_score"] = float(best["score"])
metrics_output["best_config"] = best["cfg"]
with open("model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_output, f, indent=2)

print("Saved: movie_model_hybrid.pkl, vectorizer_hybrid.pkl, scaler_hybrid.pkl")
print("Saved: movies.json, model_metrics.json")

In [ ]:
# Display final metrics
with open("model_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("=== Model Performance Metrics ===")
print(f"Movies used: {metrics['movies_used']}")
print(f"K: {metrics['k']}")
print(f"Avg neighbor similarity@k: {metrics['avg_neighbor_similarity_at_k']:.4f}")
print(f"Avg genre jaccard@k: {metrics['avg_genre_jaccard_at_k']:.4f}")
print(f"Catalog coverage@k: {metrics['catalog_coverage_at_k']:.4f}")
print(f"Precision@k: {metrics['precision_at_k']:.4f}")
print(f"Recall@k: {metrics['recall_at_k']:.4f}")
print(f"F1-score@k: {metrics['f1_score_at_k']:.4f}")
print(f"Search score: {metrics['search_score']:.4f}")
print(f"Best config: {metrics['best_config']}")

In [ ]:
# Optional: download outputs
from google.colab import files
for fname in [
    "movie_model_hybrid.pkl",
    "vectorizer_hybrid.pkl",
    "scaler_hybrid.pkl",
    "movies.json",
    "model_metrics.json",
]:
    files.download(fname)